## Transform Constructors Data (Bronze → Silver)
Steps:
1. Read constructors.csv from Bronze
2. Add ingestion timestamp
3. Select & rename required columns
4. Register table in catalog
5. Store Delta files in ADLS

In [0]:
from pyspark.sql.functions import current_timestamp, col

In [0]:
%run "/Workspace/f1-project/00-Configurations"

In [0]:
constructors_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{bronze_path}/constructors.csv")

In [0]:
constructors_df.show()

+-------------+--------------+-----------+-----------+--------------------+
|constructorId|constructorRef|       name|nationality|                 url|
+-------------+--------------+-----------+-----------+--------------------+
|            1|       mclaren|    McLaren|    British|http://en.wikiped...|
|            2|    bmw_sauber| BMW Sauber|     German|http://en.wikiped...|
|            3|      williams|   Williams|    British|http://en.wikiped...|
|            4|       renault|    Renault|     French|http://en.wikiped...|
|            5|    toro_rosso| Toro Rosso|    Italian|http://en.wikiped...|
|            6|       ferrari|    Ferrari|    Italian|http://en.wikiped...|
|            7|        toyota|     Toyota|   Japanese|http://en.wikiped...|
|            8|   super_aguri|Super Aguri|   Japanese|http://en.wikiped...|
|            9|      red_bull|   Red Bull|   Austrian|http://en.wikiped...|
|           10|   force_india|Force India|     Indian|http://en.wikiped...|
|           

In [0]:
constructors_with_ingestion_df = constructors_df.withColumn("ingestion_date", current_timestamp())

In [0]:
constructors_selected_df = constructors_with_ingestion_df.select(
    col("constructorId").alias("constructor_id"),
    col("constructorRef").alias("constructor_ref"),
    col("name").alias("constructor_name"),
    col("nationality"),
    col("ingestion_date")
)

In [0]:
constructors_selected_df.show()

+--------------+---------------+----------------+-----------+--------------------+
|constructor_id|constructor_ref|constructor_name|nationality|      ingestion_date|
+--------------+---------------+----------------+-----------+--------------------+
|             1|        mclaren|         McLaren|    British|2026-03-14 09:27:...|
|             2|     bmw_sauber|      BMW Sauber|     German|2026-03-14 09:27:...|
|             3|       williams|        Williams|    British|2026-03-14 09:27:...|
|             4|        renault|         Renault|     French|2026-03-14 09:27:...|
|             5|     toro_rosso|      Toro Rosso|    Italian|2026-03-14 09:27:...|
|             6|        ferrari|         Ferrari|    Italian|2026-03-14 09:27:...|
|             7|         toyota|          Toyota|   Japanese|2026-03-14 09:27:...|
|             8|    super_aguri|     Super Aguri|   Japanese|2026-03-14 09:27:...|
|             9|       red_bull|        Red Bull|   Austrian|2026-03-14 09:27:...|
|   

In [0]:
constructors_final_df = constructors_selected_df

In [0]:
constructors_final_df = constructors_final_df.dropDuplicates(["constructor_id"])

In [0]:
constructors_final_df.show()

+--------------+---------------+----------------+-------------+--------------------+
|constructor_id|constructor_ref|constructor_name|  nationality|      ingestion_date|
+--------------+---------------+----------------+-------------+--------------------+
|           148|    frazer_nash|     Frazer Nash|      British|2026-03-14 09:27:...|
|            31|         simtek|          Simtek|      British|2026-03-14 09:27:...|
|            85|        bellasi|         Bellasi|        Swiss|2026-03-14 09:27:...|
|           137| arzani-volpini|  Arzani-Volpini|      Italian|2026-03-14 09:27:...|
|            65|        martini|         Martini|       French|2026-03-14 09:27:...|
|            53|        toleman|         Toleman|      British|2026-03-14 09:27:...|
|           133|            hwm|             HWM|      British|2026-03-14 09:27:...|
|            78|           amon|            Amon|New Zealander|2026-03-14 09:27:...|
|           108|        epperly|         Epperly|     American|20

In [0]:
%sql
USE CATALOG f1_catalog

In [0]:
%sql
USE SCHEMA f1_silver

In [0]:
constructors_final_df.write.mode("overwrite").format("delta").saveAsTable('constructors_1')

In [0]:
%sql
SELECT * FROM f1_silver.constructors_1

constructor_id,constructor_ref,constructor_name,nationality,ingestion_date
148,frazer_nash,Frazer Nash,British,2026-03-14T09:32:49.231371Z
31,simtek,Simtek,British,2026-03-14T09:32:49.231371Z
85,bellasi,Bellasi,Swiss,2026-03-14T09:32:49.231371Z
137,arzani-volpini,Arzani-Volpini,Italian,2026-03-14T09:32:49.231371Z
65,martini,Martini,French,2026-03-14T09:32:49.231371Z
53,toleman,Toleman,British,2026-03-14T09:32:49.231371Z
133,hwm,HWM,British,2026-03-14T09:32:49.231371Z
78,amon,Amon,New Zealander,2026-03-14T09:32:49.231371Z
108,epperly,Epperly,American,2026-03-14T09:32:49.231371Z
155,hall,Hall,American,2026-03-14T09:32:49.231371Z


Write to ADLS

In [0]:
constructors_final_df.write.mode("overwrite").format("delta").save(f"{silver_path}/constructors")